In [ ]:
import rasterio

In [ ]:
with rasterio.open() as src:
    profile = src.profile.copy()
    data = src.read(1)
    
    # 1️⃣ Change existing nodata to -9999
    nodata_val = src.nodata if src.nodata is not None else -9999
    data[data == nodata_val] = -9999
    profile.update(nodata=-9999)
    
    # 2️⃣ Mask outside Italy
    mask_image, _ = mask(src, geoms, invert=False, filled=False)
    mask_image = mask_image[0].astype(bool)  # Convert 3D -> 2D, True inside Italy
    
    # Pixels outside Italy with value 0 → -9999
    outside_mask = ~mask_image
    data[outside_mask & (data == 0)] = -9999

# Save raster
with rasterio.open(output_file, 'w', **profile) as dst:
    dst.write(data, 1)